In [1]:
import cv2 
import numpy as np 
from jetbot import Camera, Robot 
from http.server import BaseHTTPRequestHandler, HTTPServer 
import threading 
import urllib.parse 
import json
# Initialize the camera and robot 
# camera = Camera.instance(width=224, height=224) 
camera = Camera.instance(width=1500, height=1500) 

robot = Robot()

In [2]:
# Function to convert BGR8 images to JPEG 
def bgr8_to_jpeg(value): 
    return cv2.imencode('.jpg', value)[1].tobytes() 


In [3]:
# HTTP request handler class 
class CameraHandler(BaseHTTPRequestHandler): 
    def do_GET(self): 
        if self.path.startswith('/camera'):
            frame = camera.value  # 1000x1000

            # Crop: behåll mitten 300 px i höjd
            cropped = frame[500:1400, :]

            self.send_response(200)
            self.send_header('Content-type', 'image/jpeg')
            self.end_headers()
            self.wfile.write(bgr8_to_jpeg(cropped))
        else: 
            query = urllib.parse.urlparse(self.path).query 
            params = urllib.parse.parse_qs(query) 
            if self.path.startswith('/set_motors'): 
                left_speed = float(params.get('left', [0])[0]) 
                right_speed = float(params.get('right', [0])[0]) 
                robot.set_motors(left_speed, right_speed) 
                self.send_response(200) 
                self.end_headers() 
                self.wfile.write(b'Motors set') 
            elif self.path.startswith('/left'): 
                speed = float(params.get('speed', [0])[0]) 
                robot.left(speed) 
                self.send_response(200) 
                self.end_headers() 
                self.wfile.write(b'Left command executed') 
            elif self.path.startswith('/right'): 
                speed = float(params.get('speed', [0])[0]) 
                robot.right(speed) 
                self.send_response(200) 
                self.end_headers() 
                self.wfile.write(b'Right command executed') 
            elif self.path.startswith('/forward'): 
                speed = float(params.get('speed', [0])[0]) 
                robot.forward(speed) 
                self.send_response(200) 
                self.end_headers() 
                self.wfile.write(b'Forward command executed') 
            elif self.path.startswith('/stop'): 
                robot.stop() 
                self.send_response(200) 
                self.end_headers() 
                self.wfile.write(b'Stop command executed')
            elif self.path.startswith('/status'):
                self.send_response(200)
                self.send_header('Content-type', 'application/json')
                self.end_headers()

                self.wfile.write(json.dumps(robot_status).encode())

In [ ]:
# Shared robot status
robot_status = {
    "see_red": False,
    "red_distance": -1,
    "colors": []
}

# Function to update robot state
def update_status(see_red=False, distance=-1, colors=None):
    global robot_status

    if colors is None:
        colors = []

    robot_status = {
        "see_red": see_red,
        "red_distance": distance,
        "colors": colors
    }

In [4]:
# Function to run the HTTP server 
def run_server(): 
    server = HTTPServer(('0.0.0.0', 8080), CameraHandler) 
    server.serve_forever() 
    
# Start the server in a separate thread 
thread = threading.Thread(target=run_server) 
thread.start() 

194.47.156.180 - - [13/May/2026 12:25:38] "GET /camera HTTP/1.1" 200 -
194.47.156.180 - - [13/May/2026 12:25:38] "GET /camera HTTP/1.1" 200 -
194.47.156.180 - - [13/May/2026 12:25:38] "GET /camera HTTP/1.1" 200 -
194.47.156.180 - - [13/May/2026 12:25:38] "GET /camera HTTP/1.1" 200 -
194.47.156.180 - - [13/May/2026 12:25:39] "GET /camera HTTP/1.1" 200 -
194.47.156.180 - - [13/May/2026 12:25:39] "GET /camera HTTP/1.1" 200 -
194.47.156.180 - - [13/May/2026 12:25:39] "GET /camera HTTP/1.1" 200 -
194.47.156.180 - - [13/May/2026 12:25:40] "GET /camera HTTP/1.1" 200 -
194.47.156.180 - - [13/May/2026 12:25:40] "GET /camera HTTP/1.1" 200 -
194.47.156.180 - - [13/May/2026 12:25:40] "GET /camera HTTP/1.1" 200 -
194.47.156.180 - - [13/May/2026 12:25:40] "GET /camera HTTP/1.1" 200 -
194.47.156.180 - - [13/May/2026 12:25:40] "GET /camera HTTP/1.1" 200 -
194.47.156.180 - - [13/May/2026 12:25:41] "GET /camera HTTP/1.1" 200 -
194.47.156.180 - - [13/May/2026 12:25:41] "GET /camera HTTP/1.1" 200 -
194.47